# Silver → Gold ETL — Star Schema

Builds the dimensional model defined in the MySQL DDL into `data_engineering.gold`,
sourcing from the three `data_engineering.silver.*_transformed` tables.

## Tables produced

| Layer | Table | Grain |
|---|---|---|
| Dimension | `dim_location` | one row per state/territory |
| Dimension | `dim_svi` | one row per state |
| Dimension | `dim_medicaid` | one row per state |
| Dimension | `dim_respondent` | one row per respondent |
| Dimension | `dim_behavior` | one row per respondent |
| Dimension | `dim_chronic_condition` | one row per respondent |
| Dimension | `dim_healthcare_access` | one row per respondent |
| Dimension | `dim_preventive_care` | one row per respondent |
| Dimension | `dim_time` | one row per (month, year, quarter) tuple |
| Fact | `fact_health_response` | one row per respondent |

## DDL-to-Delta notes

- MySQL `AUTO_INCREMENT` → surrogate keys generated via `row_number()` ordered by a stable natural key
- MySQL `TINYINT(1)` (boolean) → Delta `BOOLEAN`
- MySQL `DECIMAL(p,s)` / `SMALLINT` → cast explicitly
- Per-respondent dimensions share the *same* surrogate key value for the same respondent, so fact joins are trivial
- Delta `FOREIGN KEY` requires the referenced column to be the parent's `PRIMARY KEY`. The DDL's
  `fk_svi_location (state_code)` and `fk_medicaid_location (state_name)` would target non-PK columns
  in `dim_location`, so they are **skipped** here. The 7 fact→dim FKs are declared informationally.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType, ShortType, IntegerType, LongType

CATALOG = "data_engineering"
GOLD    = f"{CATALOG}.gold"

## 0. Create gold schema

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")
print(f"Schema ready: {GOLD}")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7123620964529464>, line 1
----> 1 spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")
      2 print(f"Schema ready: {GOLD}")

NameError: name 'GOLD' is not defined

## 1. Load silver tables

In [0]:
brfss = spark.read.table(f"{CATALOG}.silver.brfss_2024_transformed")
svi   = spark.read.table(f"{CATALOG}.silver.svi_2022_state_transformed")
med   = spark.read.table(f"{CATALOG}.silver.medicaid_expansion_clean_transformed")

print(f"BRFSS:    {brfss.count():>8,} rows x {len(brfss.columns):>3} cols")
print(f"SVI:      {svi.count():>8,} rows x {len(svi.columns):>3} cols")
print(f"Medicaid: {med.count():>8,} rows x {len(med.columns):>3} cols")

BRFSS:     457,670 rows x  73 cols
SVI:            51 rows x  16 cols
Medicaid:       54 rows x   3 cols


## 2. Generate the per-respondent surrogate key

Stable across all five per-respondent dimensions and the fact table — they all share the same key value
for a given `respondent_id`, just aliased differently per dim.

`row_number()` over a global `orderBy` does force everything to a single partition (Spark warns about it),
but we need the sequential surrogate. To avoid recomputing it five times (once per per-respondent dim),
we materialize the keyed BRFSS to a **temp Delta table** and read it back from there. Caching is not
available on serverless, so materialization is the right substitute.

In [0]:
TMP_TABLE = f"{GOLD}._tmp_brfss_keyed"

resp_window = Window.orderBy(F.col("respondent_id").cast("string"))
brfss_with_key = (brfss
    .withColumn("respondent_key", F.row_number().over(resp_window).cast(LongType()))
    .withColumn("respondent_id_bigint", F.col("respondent_id").cast(LongType()))
)

(brfss_with_key.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TMP_TABLE))

brfss_keyed = spark.read.table(TMP_TABLE)
n_brfss = brfss_keyed.count()
print(f"BRFSS keyed: {n_brfss:,} rows; respondent_key generated and materialized to {TMP_TABLE}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


BRFSS keyed: 457,670 rows; respondent_key generated and materialized to data_engineering.gold._tmp_brfss_keyed


## 3. `dim_location`

In [0]:
dim_location = (brfss
    .where(F.col("state_fips").isNotNull() & F.col("state_code").isNotNull())
    .select("state_fips", "state_code", "state_name")
    .distinct()
    .withColumn("location_key", F.row_number().over(Window.orderBy("state_fips")).cast(LongType()))
    .select(
        F.col("location_key"),
        F.col("state_fips").cast(DecimalType(5, 1)).alias("state_fips"),
        F.col("state_code").cast("string").alias("state_code"),
        F.col("state_name").cast("string").alias("state_name"),
    )
)

(dim_location.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.dim_location"))

print(f"dim_location: {spark.read.table(f'{GOLD}.dim_location').count()} rows")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_location: 53 rows


## 4. `dim_svi`

In [0]:
dim_svi = (svi
    .withColumn("svi_key", F.row_number().over(Window.orderBy("state_code")).cast(LongType()))
    .select(
        F.col("svi_key"),
        F.col("state_code").cast("string"),
        F.col("state_name").cast("string"),
        F.col("svi_overall").cast(DecimalType(6, 4)),
        F.col("svi_socioeconomic").cast(DecimalType(6, 4)),
        F.col("svi_household").cast(DecimalType(6, 4)),
        F.col("svi_minority").cast(DecimalType(6, 4)),
        F.col("svi_housing_transport").cast(DecimalType(6, 4)),
        F.col("pct_uninsured").cast(DecimalType(6, 2)),
        F.col("pct_poverty").cast(DecimalType(6, 2)),
        F.col("pct_unemployed").cast(DecimalType(6, 2)),
        F.col("pct_no_hs_diploma").cast(DecimalType(6, 2)),
        F.col("pct_elderly").cast(DecimalType(6, 2)),
        F.col("pct_disabled").cast(DecimalType(6, 2)),
        F.col("pct_minority").cast(DecimalType(6, 2)),
        F.col("pct_no_vehicle").cast(DecimalType(6, 2)),
        F.col("pct_no_internet").cast(DecimalType(6, 2)),
    )
)

(dim_svi.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.dim_svi"))

print(f"dim_svi: {spark.read.table(f'{GOLD}.dim_svi').count()} rows")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_svi: 51 rows


## 5. `dim_medicaid`

In [0]:
dim_medicaid = (med
    .withColumn("medicaid_key", F.row_number().over(Window.orderBy("state_name")).cast(LongType()))
    .select(
        F.col("medicaid_key"),
        F.col("state_name").cast("string"),
        F.col("medicaid_expansion_status").cast("string"),
        F.col("expansion_year").cast(ShortType()),
    )
)

(dim_medicaid.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.dim_medicaid"))

print(f"dim_medicaid: {spark.read.table(f'{GOLD}.dim_medicaid').count()} rows")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_medicaid: 54 rows


## 6. `dim_respondent`

In [0]:
dim_respondent = brfss_keyed.select(
    F.col("respondent_key"),
    F.col("respondent_id_bigint").alias("respondent_id"),
    F.col("sex").cast("string"),
    F.col("age_group").cast("string"),
    F.col("age_decade").cast("string"),
    F.col("race").cast("string"),
    F.col("hispanic").cast("string"),
    F.col("education").cast("string"),
    F.col("income_group").cast("string"),
    F.col("marital_status").cast("string"),
    F.col("employment").cast("string"),
    F.coalesce(F.col("num_children").cast(ShortType()), F.lit(0).cast(ShortType())).alias("num_children"),
    F.col("veteran").cast("boolean"),
    F.col("pregnant").cast("boolean"),
    F.col("food_insecure").cast("string"),
    F.col("transport_barrier").cast("boolean"),
)

(dim_respondent.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.dim_respondent"))

print(f"dim_respondent: {spark.read.table(f'{GOLD}.dim_respondent').count():,} rows")

dim_respondent: 457,670 rows


## 7. `dim_behavior`

The DDL marks `marijuana_use` and `firearm_home` as `VARCHAR(15)`, but the silver pipeline left them as raw
numeric codes (`double`). They are cast to `string` here so the column type matches the DDL; if you decide to
label-encode them upstream in silver, this cast becomes a no-op.

In [0]:
dim_behavior = brfss_keyed.select(
    F.col("respondent_key").alias("behavior_key"),
    F.col("respondent_id_bigint").alias("respondent_id"),
    F.col("exercise").cast("boolean"),
    F.col("smoke_status").cast("string"),
    F.col("ever_smoker").cast("boolean"),
    F.col("ecig_use").cast("string"),
    F.col("any_alcohol").cast("boolean"),
    F.col("binge_drinking").cast("boolean"),
    F.col("heavy_drinker").cast("boolean"),
    F.col("avg_drinks_per_day").cast(DecimalType(5, 1)),
    F.col("marijuana_use").cast("string"),    # silver kept as numeric code
    F.col("sugary_drinks").cast(DecimalType(8, 1)),
    F.col("firearm_home").cast("string"),     # silver kept as numeric code
    F.col("bmi_raw").cast(DecimalType(5, 2)),
    F.col("bmi_category").cast("string"),
    F.col("bmi_category_clean").cast("string"),
)

(dim_behavior.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.dim_behavior"))

print(f"dim_behavior: {spark.read.table(f'{GOLD}.dim_behavior').count():,} rows")

dim_behavior: 457,670 rows


## 8. `dim_chronic_condition`

In [0]:
dim_chronic_condition = brfss_keyed.select(
    F.col("respondent_key").alias("condition_key"),
    F.col("respondent_id_bigint").alias("respondent_id"),
    F.col("heart_attack").cast("boolean"),
    F.col("stroke").cast("boolean"),
    F.col("asthma").cast("boolean"),
    F.col("depression").cast("boolean"),
    F.col("copd").cast("boolean"),
    F.col("arthritis").cast("boolean"),
    F.col("cancer").cast("boolean"),
    F.col("kidney_disease").cast("boolean"),
    F.col("diabetes").cast("boolean"),
    F.col("diabetes_status").cast("string"),
    F.coalesce(F.col("condition_count").cast(DecimalType(3, 1)), F.lit(0).cast(DecimalType(3, 1))).alias("condition_count"),
)

(dim_chronic_condition.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.dim_chronic_condition"))

print(f"dim_chronic_condition: {spark.read.table(f'{GOLD}.dim_chronic_condition').count():,} rows")

dim_chronic_condition: 457,670 rows


## 9. `dim_healthcare_access`

In [0]:
dim_healthcare_access = brfss_keyed.select(
    F.col("respondent_key").alias("access_key"),
    F.col("respondent_id_bigint").alias("respondent_id"),
    F.col("has_insurance").cast("string"),
    F.col("insurance_type").cast("string"),     # silver kept as numeric code
    F.col("has_pcp").cast("string"),
    F.col("cost_barrier").cast("boolean"),
    F.col("last_checkup").cast("string"),
    F.col("last_dental").cast("string"),
)

(dim_healthcare_access.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.dim_healthcare_access"))

print(f"dim_healthcare_access: {spark.read.table(f'{GOLD}.dim_healthcare_access').count():,} rows")

dim_healthcare_access: 457,670 rows


## 10. `dim_preventive_care`

In [0]:
dim_preventive_care = brfss_keyed.select(
    F.col("respondent_key").alias("preventive_key"),
    F.col("respondent_id_bigint").alias("respondent_id"),
    F.col("mammogram_2yr").cast("boolean"),
    F.col("cervical_screen").cast("boolean"),
    F.col("colorectal_screen").cast("boolean"),
    F.col("flu_shot").cast("boolean"),
    F.col("pneumo_vaccine").cast("boolean"),
    F.col("hiv_test").cast("boolean"),
)

(dim_preventive_care.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.dim_preventive_care"))

print(f"dim_preventive_care: {spark.read.table(f'{GOLD}.dim_preventive_care').count():,} rows")

dim_preventive_care: 457,670 rows


## 11. `dim_time`

In [0]:
dim_time = (brfss
    .select(
        F.col("interview_month").cast(ShortType()).alias("interview_month"),
        F.col("interview_year").cast(ShortType()).alias("interview_year"),
        F.col("quarter").cast("string").alias("quarter"),
    )
    .distinct()
    .withColumn(
        "time_key",
        F.row_number().over(Window.orderBy("interview_year", "interview_month", "quarter")).cast(LongType()),
    )
    .select("time_key", "interview_month", "interview_year", "quarter")
)

(dim_time.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.dim_time"))

print(f"dim_time: {spark.read.table(f'{GOLD}.dim_time').count()} rows")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_time: 16 rows


## 12. `fact_health_response`

Joins keyed BRFSS to `dim_location` (on `state_code`) and `dim_time` (on the time tuple).
The five per-respondent dimensions share `respondent_key` as their PK value, so they map directly.

In [0]:
dim_loc_lkp  = spark.read.table(f"{GOLD}.dim_location").select("location_key", "state_code")
dim_time_lkp = spark.read.table(f"{GOLD}.dim_time").select(
    "time_key",
    F.col("interview_month").alias("im"),
    F.col("interview_year").alias("iy"),
    F.col("quarter").alias("q"),
)

fact = (brfss_keyed
    .join(dim_loc_lkp, on="state_code", how="left")
    .join(
        dim_time_lkp,
        on=[
            brfss_keyed.interview_month.cast(ShortType()) == dim_time_lkp.im,
            brfss_keyed.interview_year.cast(ShortType()) == dim_time_lkp.iy,
            brfss_keyed.quarter == dim_time_lkp.q,
        ],
        how="left",
    )
    .select(
        # response_key is 1:1 with respondent_key (fact has the same grain as respondent),
        # so reuse it rather than triggering a second global row_number() sort.
        F.col("respondent_key").alias("response_key"),
        F.col("respondent_id_bigint").alias("respondent_id"),
        F.col("location_key"),
        F.col("respondent_key"),
        F.col("respondent_key").alias("behavior_key"),
        F.col("respondent_key").alias("condition_key"),
        F.col("respondent_key").alias("access_key"),
        F.col("respondent_key").alias("preventive_key"),
        F.col("time_key"),
        F.col("survey_weight").cast(DecimalType(12, 6)),
        F.col("log_weight").cast(DecimalType(10, 6)),
        F.col("general_health").cast("string"),
        F.col("phys_unhealthy_days").cast(ShortType()),
        F.col("mental_unhealthy_days").cast(ShortType()),
        F.col("life_satisfaction").cast("string"),
        F.col("physhlth_bin").cast("string"),
        F.col("menthlth_bin").cast("string"),
        F.col("questionnaire_ver").cast("string"),    # silver kept as numeric code
        F.col("language").cast("string"),             # silver kept as numeric code
        F.col("cell_phone_only").cast("boolean"),
    )
)

(fact.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.fact_health_response"))

print(f"fact_health_response: {spark.read.table(f'{GOLD}.fact_health_response').count():,} rows")

fact_health_response: 457,670 rows


## 13. Informational PK / FK constraints

Delta supports `PRIMARY KEY` and `FOREIGN KEY` constraints declared `NOT ENFORCED`. They are used for
query-plan optimization (joins) and for documentation in catalog viewers.

Each `ALTER TABLE` is wrapped in `try/except` so reruns of this notebook are idempotent — the second time
through, the "constraint already exists" errors are ignored.

In [0]:
KEY_COLS = {
    "dim_location":          "location_key",
    "dim_svi":               "svi_key",
    "dim_medicaid":          "medicaid_key",
    "dim_respondent":        "respondent_key",
    "dim_behavior":          "behavior_key",
    "dim_chronic_condition": "condition_key",
    "dim_healthcare_access": "access_key",
    "dim_preventive_care":   "preventive_key",
    "dim_time":              "time_key",
    "fact_health_response":  "response_key",
}

def safe(stmt):
    try:
        spark.sql(stmt)
        print(f"  OK   {stmt[:90]}")
    except Exception as e:
        print(f"  skip {stmt[:90]}  ({type(e).__name__})")

# NOT NULL on every PK column (PK declaration requires it)
for tbl, col in KEY_COLS.items():
    safe(f"ALTER TABLE {GOLD}.{tbl} ALTER COLUMN {col} SET NOT NULL")

# Primary keys
for tbl, col in KEY_COLS.items():
    safe(f"ALTER TABLE {GOLD}.{tbl} ADD CONSTRAINT pk_{tbl} PRIMARY KEY ({col})")

# Fact → dim foreign keys (each surrogate references the corresponding dim PK)
FKS = [
    ("location_key",   "dim_location"),
    ("respondent_key", "dim_respondent"),
    ("behavior_key",   "dim_behavior"),
    ("condition_key",  "dim_chronic_condition"),
    ("access_key",     "dim_healthcare_access"),
    ("preventive_key", "dim_preventive_care"),
    ("time_key",       "dim_time"),
]
for fk_col, parent in FKS:
    safe(
        f"ALTER TABLE {GOLD}.fact_health_response ADD CONSTRAINT fk_fact_{parent.replace('dim_','')} "
        f"FOREIGN KEY ({fk_col}) REFERENCES {GOLD}.{parent}"
    )

  OK   ALTER TABLE data_engineering.gold.dim_location ALTER COLUMN location_key SET NOT NULL
  OK   ALTER TABLE data_engineering.gold.dim_svi ALTER COLUMN svi_key SET NOT NULL
  OK   ALTER TABLE data_engineering.gold.dim_medicaid ALTER COLUMN medicaid_key SET NOT NULL
  OK   ALTER TABLE data_engineering.gold.dim_respondent ALTER COLUMN respondent_key SET NOT NULL
  OK   ALTER TABLE data_engineering.gold.dim_behavior ALTER COLUMN behavior_key SET NOT NULL
  OK   ALTER TABLE data_engineering.gold.dim_chronic_condition ALTER COLUMN condition_key SET NOT
  OK   ALTER TABLE data_engineering.gold.dim_healthcare_access ALTER COLUMN access_key SET NOT NU
  OK   ALTER TABLE data_engineering.gold.dim_preventive_care ALTER COLUMN preventive_key SET NOT 
  OK   ALTER TABLE data_engineering.gold.dim_time ALTER COLUMN time_key SET NOT NULL
  OK   ALTER TABLE data_engineering.gold.fact_health_response ALTER COLUMN response_key SET NOT N
  OK   ALTER TABLE data_engineering.gold.dim_location ADD CONSTR

## 14. Validate row counts

In [0]:
import pandas as pd
rows = []
for tbl in list(KEY_COLS.keys()):
    n = spark.read.table(f"{GOLD}.{tbl}").count()
    rows.append((tbl, n))
pdf = pd.DataFrame(rows, columns=["table", "row_count"])
print(pdf.to_string(index=False))

# Drop the materialization scratch table
spark.sql(f"DROP TABLE IF EXISTS {TMP_TABLE}")
print(f"\nCleaned up {TMP_TABLE}")

                table  row_count
         dim_location         53
              dim_svi         51
         dim_medicaid         54
       dim_respondent     457670
         dim_behavior     457670
dim_chronic_condition     457670
dim_healthcare_access     457670
  dim_preventive_care     457670
             dim_time         16
 fact_health_response     457670

Cleaned up data_engineering.gold._tmp_brfss_keyed
